# Notebook 03: Cedar Policy Authorization via Amazon Verified Permissions

## Overview

Notebook 02 proved that Teleport identity flows into every tool call. This notebook
adds **policy enforcement**: the interceptor Lambda now calls Amazon Verified Permissions
(AVP) before forwarding any tool call. AVP evaluates Cedar policies that map
Teleport roles to allowed tools — no code changes required to add or change a rule.

```
tools/call arrives at interceptor
  │
  ├─ decode Teleport JWT → sub, roles
  │
  ├─ AVP IsAuthorized:
  │     principal: TeleportUser::"jeffrey.ellin@goteleport.com"
  │     action:    Action::"invoke_tool"
  │     resource:  Tool::"update_order_tool"
  │     entities:  user is member of [mcp-user, aws-personal-admin, ...]
  │
  ├─ DENY  → MCP error returned, tool Lambda never invoked
  └─ ALLOW → inject identity, forward to tool Lambda
```

### Demo highlight — live policy change
At step 9, `update_order_tool` is denied for `mcp-user`. We then add a Cedar
policy in AVP (no redeploy) and the same call immediately succeeds.

### Prerequisites
- Notebooks 01 and 02 completed
- `.env` populated with AWS credentials

## Step 1: Setup

In [ ]:
import os, json, time, io, zipfile, subprocess
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
REGION = os.environ['AWS_DEFAULT_REGION']

# Create an explicit session so boto3 reads credentials from os.environ
# rather than any internally cached credential chain from a prior run.
session = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=REGION
)

iam            = session.client('iam')
lambda_client  = session.client('lambda')
gateway_client = session.client('bedrock-agentcore-control')
avp            = session.client('verifiedpermissions')
account_id     = session.client('sts').get_caller_identity()['Account']

GATEWAY_NAME            = 'teleport-identity-demo'
TELEPORT_CLUSTER        = 'ellinj.teleport.sh'
TELEPORT_DISCOVERY      = f'https://{TELEPORT_CLUSTER}/.well-known/openid-configuration'
GATEWAY_ROLE_NAME       = 'teleport-demo-agentcore-gateway-role'
INTERCEPTOR_LAMBDA_NAME = 'teleport-demo-interceptor'
APP_NAME                = 'agentcore-gateway'

gw = next(g for g in gateway_client.list_gateways(maxResults=100)['items']
          if g['name'] == GATEWAY_NAME)
gateway_id       = gw['gatewayId']
gateway_url      = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['gatewayUrl']
gateway_role_arn = iam.get_role(RoleName=GATEWAY_ROLE_NAME)['Role']['Arn']
interceptor_lambda_arn = lambda_client.get_function(
    FunctionName=INTERCEPTOR_LAMBDA_NAME)['Configuration']['FunctionArn']

print(f'Account    : {account_id}')
print(f'Gateway    : {gateway_id}')
print(f'Interceptor: {interceptor_lambda_arn}')

## Step 2: Create AVP Policy Store

In [ ]:
AVP_POLICY_STORE_DESC = 'Teleport role → AgentCore tool access control'

resp = avp.create_policy_store(
    validationSettings={'mode': 'OFF'},
    description=AVP_POLICY_STORE_DESC
)
policy_store_id = resp['policyStoreId']
print(f'Policy store: {policy_store_id}')

## Step 3: Cedar Policies

We use validation mode `OFF` so no Cedar JSON schema is needed.
AVP still enforces the policies exactly as written — `OFF` only skips
compile-time type-checking of policy syntax against a schema.
The allow/deny runtime behaviour is unchanged.

In [ ]:
print('Validation mode is OFF — no schema required. Proceeding to policy creation.')

## Step 4: Create Cedar Policies

- `mcp-user` role → `whoami_tool` and `get_order_tool`
- `order-admin` role → `update_order_tool`

The caller holds `mcp-user` but **not** `order-admin`, so `update_order_tool` should
be denied at step 9. Step 10 grants it to `mcp-user` live — no redeploy needed.

In [ ]:
policies = [
    (
        'mcp-user-whoami',
        'mcp-user role can call whoami_tool',
        'permit(principal in TeleportRole::"mcp-user", action == Action::"invoke_tool", resource == Tool::"whoami_tool");'
    ),
    (
        'mcp-user-get-order',
        'mcp-user role can read orders',
        'permit(principal in TeleportRole::"mcp-user", action == Action::"invoke_tool", resource == Tool::"get_order_tool");'
    ),
    (
        'order-admin-update-order',
        'order-admin role can mutate orders (user does not have this role)',
        'permit(principal in TeleportRole::"order-admin", action == Action::"invoke_tool", resource == Tool::"update_order_tool");'
    ),
]

policy_ids = {}
for name, description, statement in policies:
    resp = avp.create_policy(
        policyStoreId=policy_store_id,
        definition={'static': {'description': description, 'statement': statement}}
    )
    policy_ids[name] = resp['policyId']
    print(f'  {name}: {resp["policyId"]}')

print(f'\n{len(policy_ids)} policies created')
print()
print('mcp-user    → whoami_tool, get_order_tool')
print('order-admin → update_order_tool  (caller does not hold this role → DENY expected)')

## Step 5: Grant Interceptor Lambda Permission to Call AVP

In [ ]:
interceptor_role_name = 'teleport-demo-interceptor-lambda-role'

iam.put_role_policy(
    RoleName=interceptor_role_name,
    PolicyName='AVPIsAuthorized',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': 'verifiedpermissions:IsAuthorized',
            'Resource': f'arn:aws:verifiedpermissions::{account_id}:policy-store/{policy_store_id}'
        }]
    })
)
print('AVP IsAuthorized permission granted to interceptor role')

## Step 6: Deploy Updated Interceptor Lambda

`lambda_interceptor_avp.py` adds AVP `IsAuthorized` calls before forwarding.
The policy store ID is passed via the `AVP_POLICY_STORE_ID` environment variable.

In [ ]:
with open('lambda_interceptor_avp.py', 'r') as f:
    code = f.read()

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', code)
buf.seek(0)
pkg = buf.read()

lambda_client.update_function_code(
    FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=pkg
)
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)

lambda_client.update_function_configuration(
    FunctionName=INTERCEPTOR_LAMBDA_NAME,
    Environment={'Variables': {'AVP_POLICY_STORE_ID': policy_store_id}}
)
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)

print(f'Interceptor updated with AVP_POLICY_STORE_ID={policy_store_id}')

## Step 7: Helper — Call a Tool via tsh

In [ ]:
import json as _json

def call_tool(tool_name, arguments={}):
    """Call a gateway tool via tsh mcp connect and return the parsed result."""
    msgs = [
        '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"notebook","version":"0.0.1"}}}',
        _json.dumps({
            'jsonrpc': '2.0', 'id': 2, 'method': 'tools/call',
            'params': {'name': f'TeleportDemo___{tool_name}', 'arguments': arguments}
        })
    ]
    result = subprocess.run(
        ['tsh', 'mcp', 'connect', APP_NAME],
        input='\n'.join(msgs) + '\n',
        capture_output=True, text=True, timeout=30
    )
    for line in result.stdout.splitlines():
        try:
            msg = _json.loads(line)
            if msg.get('id') == 2:
                return msg
        except Exception:
            pass
    return None

def show(tool_name, arguments={}):
    print(f'\n→ {tool_name}({arguments})')
    msg = call_tool(tool_name, arguments)
    if msg is None:
        print('  (no response)')
        return
    if 'error' in msg:
        print(f'  ✗ DENIED: {msg["error"]["message"]}')
    else:
        content = msg['result']['content'][0]['text']
        try:
            outer = _json.loads(content)
            body  = _json.loads(outer['body'])
            print(f'  ✓ ALLOWED: {_json.dumps(body, indent=4)}')
        except Exception:
            print(f'  ✓ ALLOWED: {content}')

print('Helper ready')

## Step 8: Test Allowed Calls

`whoami_tool` and `get_order_tool` are permitted for `mcp-user`.

In [ ]:
show('whoami_tool')
show('get_order_tool', {'orderId': 'ORD-42'})

## Step 9: Test Denied Call

`update_order_tool` is only permitted for `order-admin`. The caller holds `mcp-user`
but not `order-admin` — Cedar default-deny blocks it. The tool Lambda is **never invoked**.

In [ ]:
show('update_order_tool', {'orderId': 'ORD-42'})

## Step 10: Live Policy Change — Grant Access Without Redeploying

Add a Cedar policy that permits `mcp-user` to call `update_order_tool`.
No Lambda redeployment, no gateway restart — AVP evaluates it immediately.

In [ ]:
resp = avp.create_policy(
    policyStoreId=policy_store_id,
    definition={'static': {
        'description': 'DEMO: grant mcp-user update access (added live)',
        'statement': 'permit(principal in TeleportRole::"mcp-user", action == Action::"invoke_tool", resource == Tool::"update_order_tool");'
    }}
)
new_policy_id = resp['policyId']
print(f'Policy added: {new_policy_id}')
print('No Lambda redeployment needed — calling update_order_tool again...')

show('update_order_tool', {'orderId': 'ORD-42'})

## Step 11: Revert — Remove the Policy

Delete the policy added in step 10 to restore the original deny.

In [ ]:
avp.delete_policy(policyStoreId=policy_store_id, policyId=new_policy_id)
print(f'Policy {new_policy_id} removed')

show('update_order_tool', {'orderId': 'ORD-42'})

## What Just Happened

| Step | Result |
|:-----|:-------|
| `whoami_tool` | ✓ Cedar ALLOW — `mcp-user` policy matches |
| `get_order_tool` | ✓ Cedar ALLOW — `mcp-user` policy matches |
| `update_order_tool` (before) | ✗ Cedar DENY — no policy for `mcp-user` |
| Add policy in AVP console/API | — no code deployed |
| `update_order_tool` (after) | ✓ Cedar ALLOW — new policy matched |
| Remove policy | ✗ Cedar DENY again — instantly reverted |

**Two independent audit trails:**
- Teleport logs: who connected, when, from where
- AVP logs: every `IsAuthorized` decision with the matching policy ID

## What's Next

Notebook 04 wires a Strands agent to the gateway so you can drive the same
tools with natural language and see the identity + Cedar enforcement in action
end-to-end.

## Cleanup

In [ ]:
from botocore.exceptions import ClientError

# Delete all live policies then the store (skip gracefully if already gone)
try:
    live = avp.list_policies(policyStoreId=policy_store_id).get('policies', [])
    for p in live:
        try:
            avp.delete_policy(policyStoreId=policy_store_id, policyId=p['policyId'])
            print(f'  Deleted policy: {p["policyId"]}')
        except ClientError:
            pass
    avp.delete_policy_store(policyStoreId=policy_store_id)
    print(f'Deleted policy store: {policy_store_id}')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f'Policy store {policy_store_id} already deleted — skipping')
    else:
        raise

# Roll interceptor back to identity-only version (no AVP)
with open('lambda_interceptor.py', 'r') as f:
    code = f.read()
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', code)
buf.seek(0)
lambda_client.update_function_code(FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=buf.read())
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
lambda_client.update_function_configuration(
    FunctionName=INTERCEPTOR_LAMBDA_NAME, Environment={'Variables': {}}
)
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
print('Interceptor rolled back to identity-only')